In [ ]:
import h5py
import numpy as np
import os

In [ ]:
data_dir = ... # enter directory here where .h5 files are located
file = ... # enter file name here .h5

In [ ]:
def get_ch(nw=0, el=0):
    return np.where(np.logical_and(mapping_matrix[:,0] == nw, mapping_matrix[:,1] == el))[0]
def get_nw_el(ch):
    return mapping_matrix[ch]

In [ ]:

# **Load HDF5 File**

filename = os.path.join(data_dir, file)
print(f"📂 Loading file: {filename}")

with h5py.File(filename, "r") as f:
    # **Print metadata**
    print("📌 Metadata:")
    for key, value in f["metadata"].attrs.items():
        print(f"  {key}: {value}")
    fs = f["metadata"].attrs["fs"]

    # **Print dataset info**
    print("\n📂 Datasets:")
    for group in f.keys():
        print(f"  📁 {group}")
        for key in f[group].keys():
            print(f"    📄 {key}: shape={f[group][key].shape}, dtype={f[group][key].dtype}")

    # **Load spike data**
    spike_channels = f["spike_data/spike_channels"][:]
    spike_times = f["spike_data/spike_times"][:]
    spike_waveforms = f["spike_data/spike_waveforms"][:]

    # **Load stimulation data (if exists)**
    if "stimulation_data" in f:
        stim_times = f["stimulation_data/stim_times"][:]
        stim_triggers = f["stimulation_data/stim_triggers"][:]
        stim_amplitudes = f["stimulation_data/stim_amplitudes"][:]
        stim_matrix = f["stimulation_data/stim_matrix"][:]
    else:
        stim_times = stim_triggers = stim_amplitudes = stim_matrix = None

    if "env_data" in f:
        try: 
            env_times = f["env_data/env_times"][:]
            env_mea_data = f["env_data/env_mea_data"][:]
            env_reservoir_data = f["env_data/env_reservoir_data"][:]
        except Exception as e:
            pass

    # **Load Electrode Mapping Matrix**
    if "electrode_mapping/spike_ch_to_network_electrode_mapping_matrix" in f:
        mapping_matrix = f["electrode_mapping/spike_ch_to_network_electrode_mapping_matrix"][:]
        print("\n📌 Mapping Matrix Shape:", mapping_matrix.shape)

    if "stimulation_parameters" in f:
        stim_params = f["stimulation_parameters"].attrs
        print("\n📌 Stimulation Parameters:")
        for key, value in stim_params.items():
            print(f" {key}: {value}")

# **Print First 10 Spike Events**
print("\n📊 First 10 spike events:")
print("Times:", spike_times[:10])
print("Channels:", spike_channels[:10])

# **Check if stimulation data exists**
if stim_times is not None:
    print("\n📊 First 10 stim events:")
    print("Stim Times:", stim_times[:10])
    print("Stim Triggers:", stim_triggers[:10])
    print("Stim Amplitudes:", stim_amplitudes[:10])
    print("Stim Matrix (first 10 rows):")
    print(stim_matrix[:10, :])
